In [ ]:
!pip install transformers datasets accelerate scikit-learn pyarrow

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving training_data.parquet to training_data.parquet


In [ ]:
import pandas as pd

df = pd.read_parquet("training_data.parquet")
print(df.head())

       source           case_id court_level court_name  year case_type   act  \
0  High Court  GAHC040011972015  High Court   Delhi HC  2015      <NA>  <NA>   
1  High Court  GAHC040011912015  High Court   Delhi HC  2015      <NA>  <NA>   
2  High Court  GAHC040010482015  High Court   Delhi HC  2015      <NA>  <NA>   
3  High Court  GAHC040006232014  High Court   Delhi HC  2015      <NA>  <NA>   
4  High Court  GAHC040011502014  High Court   Delhi HC  2015      <NA>  <NA>   

  section                                              title  \
0    <NA>  IA./62/2015 of M/S CAPITAL ENTERPRISES Vs M/S ...   
1    <NA>  IA./51/2015 of SHRI NYABOM TASAR Vs THE STATE ...   
2    <NA>  WP(C)/188/2015 of SHRI DOHU TANIA Vs THE STATE...   
3    <NA>  WP(C)/144/2014 of SRI BULLO TAJO Vs THE STATE ...   
4    <NA>  WP(C)/65/2014 of SHRI KOMDUK LOYA Vs THE DEPUT...   

                                         description  ... is_criminal  \
0  BEFORE THE HON'BLE MRS(DR.) JUSTICE INDIRA SHA...  ...    

In [ ]:
# Assuming df needs to be in its initial state with 'title', 'description', 'final_label'
# to run this cell correctly. If df is already processed, reloading is necessary.
import pandas as pd
if 'title' not in df.columns or 'description' not in df.columns or 'final_label' not in df.columns:
    print("Reloading dataframe as original columns (title, description, final_label) are missing.")
    df = pd.read_parquet("training_data.parquet")

# Combine text fields
df['text'] = df['title'].fillna('') + " " + df['description'].fillna('')

# Drop missing labels
df = df.dropna(subset=['final_label'])

# Encode original labels (0,1,2)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['labels'] = le.fit_transform(df['final_label'])

# 🔥 Convert to binary (ADR = 1, others = 0)
df['binary_label'] = df['labels'].apply(lambda x: 1 if x == 1 else 0)

# Keep only required columns
df = df[['text', 'binary_label']]
df = df.dropna()

# 🔥 BALANCE DATASET (CRITICAL)
df_adr = df[df['binary_label'] == 1]
df_non = df[df['binary_label'] == 0]

df_non = df_non.sample(n=len(df_adr)*2, random_state=42)

df = pd.concat([df_adr, df_non]).sample(frac=1, random_state=42)

print("Balanced distribution:\n", df['binary_label'].value_counts())

Reloading dataframe as original columns (title, description, final_label) are missing.
Balanced distribution:
 binary_label
0    76168
1    38084
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    stratify=df['binary_label'],
    random_state=42
)

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "nlpaueb/legal-bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were new

In [ ]:
for name, param in model.named_parameters():
    if "encoder.layer.10" in name or "encoder.layer.11" in name or "classifier" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

In [ ]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=160
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/102826 [00:00<?, ? examples/s]

Map:   0%|          | 0/11426 [00:00<?, ? examples/s]

In [ ]:
train_dataset = train_dataset.rename_column("binary_label", "labels")
val_dataset = val_dataset.rename_column("binary_label", "labels")

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary'
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    fp16=True
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.013223,0.019369,0.994661,0.991945,0.997875,0.986086
2,0.009601,0.008556,0.998425,0.997639,0.996855,0.998425


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=12854, training_loss=0.026697559442107544, metrics={'train_runtime': 890.8581, 'train_samples_per_second': 230.847, 'train_steps_per_second': 14.429, 'total_flos': 1.69091608615296e+16, 'train_loss': 0.026697559442107544, 'epoch': 2.0})

In [ ]:
trainer.evaluate()

{'eval_loss': 0.008555784821510315,
 'eval_accuracy': 0.9984246455452477,
 'eval_f1': 0.9976390346274921,
 'eval_precision': 0.9968545216251639,
 'eval_recall': 0.9984247834077186,
 'eval_runtime': 27.9173,
 'eval_samples_per_second': 409.28,
 'eval_steps_per_second': 25.611,
 'epoch': 2.0}

In [ ]:
trainer.save_model("legalbert-adr-model")
tokenizer.save_pretrained("legalbert-adr-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('legalbert-adr-model/tokenizer_config.json',
 'legalbert-adr-model/tokenizer.json')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

trainer.save_model("/content/drive/MyDrive/legalbert-adr-model")
tokenizer.save_pretrained("/content/drive/MyDrive/legalbert-adr-model")

Mounted at /content/drive


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/legalbert-adr-model/tokenizer_config.json',
 '/content/drive/MyDrive/legalbert-adr-model/tokenizer.json')

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("legalbert-adr-model")
tokenizer = AutoTokenizer.from_pretrained("legalbert-adr-model")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    pred = outputs.logits.argmax().item()

    return "ADR Suitable" if pred == 1 else "Not ADR Suitable"

In [ ]:
print(predict("The dispute can be resolved through mediation and mutual settlement"))

ADR Suitable
